---
format: gfm
execute:
  enabled: false      # never execute on render; use stored outputs
---

## CarpeDIAMS

A package for deconvoluting dia/aif/msn data into pseudo-dda data. Builds on the foundation of corrdec and ms-dial but integrates new clustering methods of ms2 data and more sophisticated filtering methods for the deconvolution.

It is all about catching the DIA patterns in our MS data.

Three types of input can be used to generate an mgf of ms2 data:
* An MsExperiment object from xcms
* A mzml directory path and a target list of peak positions (mz and rt)
* mzml file(s) which are submitted to peak picking and then processed by the carpediams workflow


### How it works

As with corrdec and msdial the algorithm simply finds ms2 fragments that correlate with 

In [1]:
library(Spectra)
library(xcms)

Loading required package: S4Vectors

Loading required package: stats4

Loading required package: BiocGenerics


Attaching package: ‘BiocGenerics’


The following objects are masked from ‘package:stats’:

    IQR, mad, sd, var, xtabs


The following objects are masked from ‘package:base’:

    anyDuplicated, aperm, append, as.data.frame, basename, cbind,
    colnames, dirname, do.call, duplicated, eval, evalq, Filter, Find,
    get, grep, grepl, intersect, is.unsorted, lapply, Map, mapply,
    match, mget, order, paste, pmax, pmax.int, pmin, pmin.int,
    Position, rank, rbind, Reduce, rownames, sapply, saveRDS, setdiff,
    table, tapply, union, unique, unsplit, which.max, which.min



Attaching package: ‘S4Vectors’


The following object is masked from ‘package:utils’:

    findMatches


The following objects are masked from ‘package:base’:

    expand.grid, I, unname


Loading required package: BiocParallel


This is xcms version 4.4.0 



Attaching package: ‘xcms’


The following ob

In [2]:
devtools::load_all("/faststorage/project/forensics_TOFscreenings/06_integrated_workflow/modules/shared/carpeDIAMS")

ℹ Loading carpeDIAMS


In [3]:
library(tidyr)
library(dplyr)
library(purrr)
library(tibble)
library(readr)


Attaching package: ‘tidyr’


The following object is masked from ‘package:S4Vectors’:

    expand



Attaching package: ‘dplyr’


The following objects are masked from ‘package:xcms’:

    collect, groups


The following objects are masked from ‘package:S4Vectors’:

    first, intersect, rename, setdiff, setequal, union


The following objects are masked from ‘package:BiocGenerics’:

    combine, intersect, setdiff, union


The following objects are masked from ‘package:stats’:

    filter, lag


The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union




In [4]:
files <- list.files("/faststorage/project/forensics_TOFscreenings/BACKUP/TOF2_stoffer_compassxport", recursive = TRUE, full.names = TRUE)

tmp_folder <- "/faststorage/project/forensics_TOFscreenings/06_integrated_workflow/results_all/drugs_annotation/tmp"

df <- tibble::tibble(
  file = files,
  sample_name = basename(files),
  file_size = file.size(files)
) |> 
mutate(
    peak_picked = paste0(tmp_folder, "/", gsub(".mzML$", "_peaks.rds", sample_name)),
    integration_ready = paste0(tmp_folder, "/", "integration_ready.rds"),
    integration1 = "#"
)

library(stringr)
TOF2_DB <- readr::read_csv("/faststorage/project/forensics_TOFscreenings/06_integrated_workflow/results_all/drugs_annotation/TOF-2_DB.csv")

COMPOUND_LIST <- readr::read_csv("/faststorage/project/forensics_TOFscreenings/06_integrated_workflow/results_all/drugs_annotation/compound_file_mapping.csv")

normalize <- function(x) {
  x |>
    tolower() |>
    str_replace_all("[\\-_\\. ]", "")
}

norm_db    <- normalize(TOF2_DB$name)
norm_files <- normalize(COMPOUND_LIST$basename)

# For each db name, find which filenames contain it
match_idx <- sapply(norm_db, function(db) {
  which(str_detect(norm_files, fixed(db)))
})

result <- data.frame(
  db_idx   = rep(seq_along(norm_db), lengths(match_idx)),
  file_idx = unlist(match_idx)
)

# Join in the actual columns
result <- data.frame(
  db_idx   = rep(seq_along(norm_db), lengths(match_idx)),
  file_idx = unlist(match_idx)
) |>
  mutate(
    TOF2_DB[db_idx, ],
    COMPOUND_LIST[file_idx, ]
  )

missing_frags <- result  |> 
  # mutate(
  #   `QI 1` = as.numeric(`QI 1`), 
  #   `QI 2` = as.numeric(`QI 2`), 
  #   `QI 3` = as.numeric(`QI 3`))  |> 
  select(`QI 1`, `QI 2`, `QI 3`) |> 
  map_dfc(is.na) |> 
  as.matrix() |> rowSums()

result <- result[missing_frags<3, ]



library(Spectra)
library(BiocParallel)
register(SerialParam())

mzml_path <- "/faststorage/project/forensics_TOFscreenings/BACKUP/TOF2_stoffer_compassxport/indkoering_forsoeg/indkoering_stoffer/gruppe_indkoering/"
origin_path <- "O:/HE_IFR-Rka/Instrumentdata/toks/UPLC-TOF/UPLC-TOF-2/indkoering_forsoeg/indkoering_stoffer/gruppe_indkoering/"
result_existing <- result |> mutate(mzml = gsub(origin_path, mzml_path, file), mzml = gsub(".d$", ".mzML", mzml))
result_existing <- result_existing |> filter(file.exists(mzml))

compounds <- result_existing$name |> unique()
files <- result_existing |> filter(name %in% compounds[1:5]) 


Rows: 951 Columns: 25
── Column specification ────────────────────────────────────────────────────────────────────────────────────
Delimiter: ","
chr (12): formula, name, CAS, QI 1, QI 2, QI 3, QI 1 min, comment3, comment4...
dbl (12): m/z, rt, comment 1, comment 2, minimum area, indivSigma, indivMass...
lgl  (1): relativeRetentiontime

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.
Rows: 2505 Columns: 3
── Column specification ────────────────────────────────────────────────────────────────────────────────────
Delimiter: ","
chr (3): compound, basename, file

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.


In [5]:
sps <- Spectra(files$mzml[1], backend = MsBackendMzR())
sps@backend@spectraData@listData$msLevel <- as.integer((Spectra::scanIndex(sps) %% 2) + 1)

In [6]:
library(MsExperiment); library(xcms); library(MsExperiment); library(Spectra)

cwp        <- CentWaveParam(
  peakwidth = c(2,30),
  snthresh  = 6,
  ppm       = 12,
  prefilter = c(3, 1000),
  mzdiff    = 0.01)

mse <- MsExperiment()
spectra(mse) <- sps

sampleData(mse) <- DataFrame(sample_name = unique(dataOrigin(sps)))   # MUST equal dataOrigin
mse <- linkSampleData(mse, with = "sampleData.sample_name = spectra.dataOrigin")
xdata <- findChromPeaks(mse, cwp)

Loading required package: ProtGenerics


Attaching package: ‘ProtGenerics’


The following object is masked from ‘package:stats’:

    smooth



[----------------------------------------------------------] 0/1 (  0%) in  0s

[==========================================================] 1/1 (100%) in 39s




In [73]:
library(data.table)

fd <- chromPeaks(xdata)
rownames(fd) <- rownames(chromPeakData(xdata))
spectra_df <- spectra(xdata)@backend@spectraData@listData |> bind_cols()

In [8]:
Ms2PseudoRaw2 <- function (single_sample, spectra_object, output_dir = "tmp/", 
    maxDiff = 0.01, buffer = 50L, return_spectra = FALSE) 
{
    filename <- single_sample$filename[1]
    sample_id <- single_sample$sample[1]
    min_max_scan <- c(-buffer, buffer) + range(single_sample$scan_indices)
    b <- Spectra::filterAcquisitionNum(Spectra::filterDataOrigin(spectra_object, 
        filename), min_max_scan[1]:min_max_scan[2])
    rm(spectra_object)
    gc()
    if (length(b) == 0) 
        return(NULL)
    pks_list <- Spectra::peaksData(b)
    n_peaks <- vapply(pks_list, nrow, integer(1))
    pks_matrix <- do.call(rbind, pks_list)
    all_ms <- tibble::tibble(mz = pks_matrix[, 1], int = pks_matrix[, 
        2], scan = rep(Spectra::acquisitionNum(b), n_peaks), 
        mslevel = rep(Spectra::msLevel(b), n_peaks))
    rm(b, pks_list, pks_matrix)
    gc()
    ms2_pool <- dplyr::arrange(dplyr::select(dplyr::mutate(dplyr::filter(all_ms, 
        mslevel == 2), sample = sample_id), sample, scan, mz, 
        int), scan)
    arrow::write_parquet(ms2_pool, file.path(output_dir, paste0("sample_", 
        sample_id, "_pool.parquet")))
    ms1_all <- all_ms[all_ms$mslevel == 1, c("mz", "int", "scan")]
    rm(all_ms)
    gc()
    feature_list <- split(single_sample, single_sample$feature)
    feature_ranges <- data.table::as.data.table(dplyr::summarize(dplyr::group_by(single_sample, 
        feature), mz_target = mz[1], mz_lo = mz[1] - maxDiff, 
        mz_hi = mz[1] + maxDiff, scan_lo = min(scan_indices) - 
            buffer, scan_hi = max(scan_indices) + buffer, apex_scan = apex_scan_index[1], 
        .groups = "drop"))
    feature_ranges[, `:=`(scan_lo_num = as.numeric(scan_lo), 
        scan_hi_num = as.numeric(scan_hi))]
    data.table::setkey(feature_ranges, mz_lo, mz_hi)
    ms1_dt <- data.table::as.data.table(ms1_all)
    ms1_dt[, `:=`(mz_start = mz, mz_end = mz)]
    matched <- data.table::foverlaps(ms1_dt, feature_ranges, 
        by.x = c("mz_start", "mz_end"), by.y = c("mz_lo", "mz_hi"), 
        nomatch = NULL)
    matched <- matched[scan >= scan_lo & scan <= scan_hi]
    pointer_results <- bind_rows(matched[, .(ms1_int = sum(int), 
        mz = mean(mz)), by = .(feature, scan, mz_target, apex_scan)][, 
        `:=`(sample, sample_id)])
    arrow::write_parquet(pointer_results, file.path(output_dir, 
        paste0("sample_", sample_id, "_pointers.parquet")))
    rm(ms1_dt, matched)
    gc()
    message("Finished Sample ", sample_id)
    if (return_spectra) {
        return(list(pool = ms2_pool, pointer = pointer_results))
    }
}

In [74]:
tmp_dir <- "/faststorage/project/forensics_TOFscreenings/06_integrated_workflow/modules/shared/carpeDIAMS/tmp"

In [76]:
a <- fd |> as.data.frame() |> split(1:nrow(fd))
names(a) <- rownames(fd)
scan_info <- extractScanInfo(feature_peaks = a, filenames = fileNames(xdata), spectra_df = spectra_df)
spectra <- Ms2PseudoRaw2(
    single_sample = scan_info,
    spectra_object = spectra(xdata),
    maxDiff = 0.005,
    output_dir = tmp_dir,
    return_spectra = TRUE
)

Finished Sample 1



In [77]:
stringr::str_split(string = "case_1_2", "_")

[[1]]
[1] "case" "1"    "2"

In [ ]:
features <- rownames(fd)

a <- clean_pseudo_spectra(
  targets        = c("CP2341"),
  file_directory = tmp_dir,
  fd             = fd,
  ms1_ppm        = 10,
  buffer         = 50L,
  fdr_count      = 30L,
  cor_count      = 50L,
  min_count      = 2L,
  report         = FALSE,
  mgf_dir        = NULL,
  progress       = TRUE
)


  [1/1] CP2341         


In [ ]:
a = get_raw_spec(feature_id = "CP2341",file_directory = tmp_dir)
print(a)

       sample feature       mz fragment_mz   int ms1_scan ms2_scan ms1_int
        <num>  <char>    <num>       <num> <num>    <int>    <int>   <num>
    1:      1  CP2341 61.02926    44.93030   404        2        3     235
    2:      1  CP2341 61.02926    44.94308   272        2        3     235
    3:      1  CP2341 61.02926    44.94879   260        2        3     235
    4:      1  CP2341 61.02926    44.96141   248        2        3     235
    5:      1  CP2341 61.02926    44.97400   236        2        3     235
   ---                                                                    
11990:      1  CP2341 61.02926    61.04062  2400      120      121     696
11991:      1  CP2341 61.02926    61.88616   272      120      121     696
11992:      1  CP2341 61.02926    61.92665   296      120      121     696
11993:      1  CP2341 61.02926    62.37306    88      120      121     696
11994:      1  CP2341 61.02926    62.99213   412      120      121     696
       feature_id
       

In [71]:
fd <- as.data.frame(fd)
fd

fd
<dbl>
111.0420
173.0440
193.1211
237.0673
255.1204
279.1578
287.0515
301.1406
301.2136


In [66]:
targets = "CP2341"
file_directory = tmp_dir
fd               = fd
con              = NULL
ms1_ppm          = 10
grouper          = ppm_upgma(ppm = 30)
buffer           = 50L
fdr_count        = 30L
cor_count        = 100L
min_count        = 10L
mgf_dir          = NULL
fetch_chunk_size = 200L
progress         = TRUE

own_con <- is.null(con)
if (own_con) {
  con <- setup_connection(file_directory)
  #on.exit(DBI::dbDisconnect(con, shutdown = TRUE), add = TRUE)
}

fd_df <- NULL
if (!is.null(fd)) {
  fd_df <- as.data.frame(fd)
  if ("feature_id" %in% colnames(fd_df)) rownames(fd_df) <- fd_df$feature_id
}
print(fd_df)
has_fd <- !is.null(fd_df)

total <- length(targets)
done  <- 0L
fetch_chunks <- split(targets, ceiling(seq_along(targets) / fetch_chunk_size))

results <- data.table::rbindlist(lapply(fetch_chunks, function(chunk_ids) {
  raw <- batch_get_feature_data(chunk_ids, con, chunk_size = length(chunk_ids))
  if (nrow(raw) == 0) {
    done <<- done + length(chunk_ids)
    return(NULL)
  }
  by_feature <- split(raw, by = "feature")

  data.table::rbindlist(lapply(chunk_ids, function(target) {
    done <<- done + 1L
    if (progress) cat(sprintf("\r  [%d/%d] %s         ", done, total, target))

    dt <- by_feature[[target]]
    if (is.null(dt) || nrow(dt) == 0) return(NULL)
    print(dt)

    mz_target <- if (has_fd && target %in% rownames(fd_df)) fd_df[target, "mzmed"]
                  else dt$mz[1]
    print(mz_target)
    if (length(mz_target) != 1 || is.na(mz_target)) return(NULL)
    rt_target <- if (has_fd && target %in% rownames(fd_df)) fd_df[target, "rtmed"]
                  else NA_real_

    print(rt_target)
    out <- tryCatch(
      process_pseudo_spec(dt, mz_target = mz_target,
                          grouper = grouper, buffer = buffer,
                          min_count = min_count),
      error = function(e) NULL
    )
    if (is.null(out) || nrow(out$correlations) == 0) return(NULL)

    pseudo_spec <- select_pseudo_spec(out$correlations, mz_target,
                                      ms1_ppm, cor_count, fdr_count)
    if (nrow(pseudo_spec) == 0) return(NULL)

    if (report) {
      plots <- Filter(Negate(is.null),
                      .pseudo_spec_plots(out$grouped_signal, out$correlations,
                                          pseudo_spec, target))
      if (length(plots)) {
        if (requireNamespace("cowplot", quietly = TRUE)) {
          print(cowplot::plot_grid(plotlist = plots, ncol = 2))
        } else {
          for (p in plots) print(p)
        }
      }
    }

    if (!is.null(mgf_dir)) {
      write_simple_mgf(
        pseudo_spec, mz_target = mz_target, feature_id = target,
        path = file.path(mgf_dir, paste0(target, ".mgf")),
        rt = rt_target
      )
    }

    n_samples_per_group <- out$grouped_signal[
      group %in% pseudo_spec$group,
      .(n_samples = data.table::uniqueN(sample)),
      by = group
    ]
    pseudo_spec <- merge(pseudo_spec, n_samples_per_group,
                          by = "group", all.x = TRUE)

    data.table::data.table(
      mz_g2           = pseudo_spec$fragment_mz,
      grouped_mz      = as.integer(pseudo_spec$group),
      mz              = pseudo_spec$fragment_mz,
      total_intensity = pseudo_spec$int,
      n_samples       = as.integer(pseudo_spec$n_samples),
      count           = as.integer(pseudo_spec$count),
      r               = pseudo_spec$correlation,
      t_stat          = NA_real_,
      p_value         = pseudo_spec$p_value,
      fdr             = pseudo_spec$fdr,
      feature_id      = target
    )
  }), fill = TRUE)
}), fill = TRUE)

if (progress) cat("\n")
results

DBI::dbDisconnect(con, shutdown = TRUE)

               mz      mzmin      mzmax         rt      rtmin     rtmax
CP0001  111.04198  111.04144  111.04228   6.160980   2.163000  15.48300
CP0002  173.04400  173.04260  173.04491   8.161980   6.160980  16.88400
CP0003  193.12106  193.12023  193.12221   8.161980   2.163000  16.68402
CP0004  237.06729  237.06633  237.06756   4.162002   2.163000  10.16100
CP0005  255.12043  255.11846  255.12211   8.161980   2.163000  14.15898
CP0006  279.15784  279.15687  279.15956   8.161980   2.163000  16.08300
CP0007  287.05146  287.05050  287.05244   6.160980   2.163000  10.16100
CP0008  301.14057  301.13925  301.14204   4.162002   2.163000  15.88302
CP0009  301.21364  301.21105  301.21505   4.162002   2.163000  16.28298
CP0010  305.24459  305.24339  305.24575   4.162002   2.163000  16.08300
CP0011  339.14367  339.14332  339.14396   4.162002   2.163000  16.08300
CP0012  351.28975  351.28759  351.29118   6.160980   2.163000  10.16100
CP0013  381.22730  381.22517  381.23181   8.161980   2.163000  1

<0 x 0 matrix>

In [65]:
print(results)

Null data.table (0 rows and 0 cols)


In [ ]:
b <- sps |>  |> 
    peaksData()

n_peaks  <- vapply(b, nrow, integer(1))
pks_matrix <- do.call(rbind, b)
print(sum(pks_matrix[, 2]))

b <- sps |> 
    Spectra::filterAcquisitionNum() |> 
    peaksData()

n_peaks  <- vapply(b, nrow, integer(1))
pks_matrix <- do.call(rbind, b)
print(sum(pks_matrix[, 2]))


[1] 6467674105
[1] 5135526144
